# DINOv2: Self-Supervised Computer Vision Models by Meta AI

Use the official pinned DINOv2 Small checkpoint on CPU with native Transformers. Run every maintained cell in order. The public COCO example photograph and its mirror illustrate image/patch features, not a held-out accuracy benchmark. Frozen features need a task-specific head for labels, depth or masks. No training, rented GPU or paid API is required. Install the pinned dependencies shown in the setup cell before starting. First run downloads the model weights; no weights or outputs are committed.

In [ ]:
import io, hashlib, tempfile, gc
from pathlib import Path
from urllib.request import urlopen
import numpy as np
import torch
from PIL import Image, ImageOps
from transformers import AutoImageProcessor, Dinov2Model
from huggingface_hub import snapshot_download
# python -m pip install torch==2.10.0 transformers==5.22.0 pillow==12.1.1
torch.set_num_threads(4)
torch.manual_seed(24)
print('Device: CPU; native Transformers DINOv2 backbone, no API key.')

In [ ]:
MODEL_ID = 'facebook/dinov2-small'
REVISION = 'ed25f3a31f01632728cabb09d1542f84ab7b0056'
checkpoint = snapshot_download(MODEL_ID, revision=REVISION, token=False,
    allow_patterns=['config.json','preprocessor_config.json','model.safetensors'])
processor = AutoImageProcessor.from_pretrained(checkpoint,local_files_only=True,trust_remote_code=False)
model = Dinov2Model.from_pretrained(checkpoint,local_files_only=True,
    trust_remote_code=False,dtype=torch.float32,attn_implementation='eager').eval()
assert model.config.hidden_size == 384 and model.config.patch_size == 14
assert model.config.num_hidden_layers == 12 and not model.training
weight_path = Path(checkpoint)/'model.safetensors'
weight_hash = hashlib.sha256(weight_path.read_bytes()).hexdigest()
parameter_count = sum(p.numel() for p in model.parameters())
print('Model:',MODEL_ID)
print('Revision:',REVISION)
print('Parameters:',parameter_count)
print('Safetensors SHA256:',weight_hash)

In [ ]:
IMAGE_URL = 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/coco_sample.png'
IMAGE_SHA256 = 'cf6f3c4befa148732c7453e0de5afab00f682427435fead2d88b07a9615cdac2'
with urlopen(IMAGE_URL,timeout=60) as response:
    image_bytes = response.read()
assert hashlib.sha256(image_bytes).hexdigest() == IMAGE_SHA256
image = Image.open(io.BytesIO(image_bytes)).convert('RGB')
images = [image,ImageOps.mirror(image)]
inputs = processor(images=images,return_tensors='pt')
pixels = inputs['pixel_values']
assert pixels.shape == (2,3,224,224) and torch.isfinite(pixels).all()
assert pixels.dtype == torch.float32
print('Original image size:',image.size)
print('Image SHA256:',IMAGE_SHA256)
print('Preprocessed original and mirrored image:',tuple(pixels.shape))

In [ ]:
with torch.inference_mode():
    output = model(**inputs,output_hidden_states=True)
tokens = output.last_hidden_state
image_features = output.pooler_output
patch_features = tokens[:,1:]
assert tokens.shape == (2,257,384)
assert image_features.shape == (2,384) and patch_features.shape == (2,256,384)
assert torch.isfinite(tokens).all()
torch.testing.assert_close(image_features,tokens[:,0],atol=0,rtol=0)
print('All tokens:',tuple(tokens.shape))
print('Image-level features:',tuple(image_features.shape))
print('Patch features:',tuple(patch_features.shape),'; 16 x 16 patches per image')
print('These are feature vectors, not class labels, depth values or segmentation masks.')

In [ ]:
# Independently compute the final layer normalization from its definition.
hidden = output.hidden_states[-1]
mean = hidden.mean(dim=-1,keepdim=True)
variance = ((hidden-mean)**2).mean(dim=-1,keepdim=True)
manual = (hidden-mean)/torch.sqrt(variance+model.config.layer_norm_eps)
manual = manual*model.layernorm.weight + model.layernorm.bias
torch.testing.assert_close(tokens,manual,atol=2e-5,rtol=2e-5)
with torch.inference_mode():
    single = model(pixel_values=pixels[:1]).last_hidden_state
torch.testing.assert_close(single,tokens[:1],atol=2e-5,rtol=2e-5)
# Compute the two-view cosine both with Torch and independent NumPy arithmetic.
vectors = image_features.detach().numpy().astype(np.float64)
cosine = np.dot(vectors[0],vectors[1])/(np.linalg.norm(vectors[0])*np.linalg.norm(vectors[1]))
torch_cosine = torch.nn.functional.cosine_similarity(image_features[:1],image_features[1:]).item()
np.testing.assert_allclose(cosine,torch_cosine,atol=1e-6)
assert -1 <= cosine <= 1
print('Original/mirror feature cosine:',round(float(cosine),6))
print('Manual layer normalization, CLS extraction and single/batch parity pass.')

In [ ]:
# Check a real local save/reload without writing to the source repository.
before = {name:hashlib.sha256(p.detach().numpy().tobytes()).hexdigest()
          for name,p in model.named_parameters()}
with tempfile.TemporaryDirectory(prefix='tdb-dinov2-') as folder:
    model.save_pretrained(folder,safe_serialization=True)
    restored = Dinov2Model.from_pretrained(folder,local_files_only=True,
        trust_remote_code=False,dtype=torch.float32,attn_implementation='eager').eval()
    with torch.inference_mode():
        reloaded = restored(pixel_values=pixels[:1]).last_hidden_state
    torch.testing.assert_close(reloaded,single,atol=0,rtol=0)
    after = {name:hashlib.sha256(p.detach().numpy().tobytes()).hexdigest()
             for name,p in model.named_parameters()}
    assert before == after
    assert all(p.grad is None for p in model.parameters())
    del restored
assert hashlib.sha256(weight_path.read_bytes()).hexdigest() == weight_hash
del model,output
gc.collect()
print('Saved/reloaded features match exactly; model parameters and checkpoint remain unchanged.')

## Original source appendix — not executable

All original installation and four-model-loading blocks are retained verbatim. The maintained example above uses only the Small backbone, rather than loading four models at once.

### Original block 1

```text
git clone https://github.com/facebookresearch/dinov2.git
```

### Original block 2

```text
conda env create -f conda.yaml
conda activate dinov2
```

### Original block 3

```text
pip install -r requirements.txt
```

### Original block 4

```text
import torch
dinov2_vits14 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
dinov2_vitb14 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
dinov2_vitl14 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14')
dinov2_vitg14 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitg14')
```

### Original repository feature card

The original GitHub repository card is preserved here, not reused as the public article cover. Its displayed repository statistics are historical.

![Original DINOv2 repository card](assets/dinov2-original/github-card.webp)